This notebook allows the user to change defacing parameters to see impacts on the performance. Other than dependencies in `requirements.txt`, also install ipykernel, ipympl and ipywidgets.

**Interactive Mode**\
Create a config file with information regarding your input data, as instructed in the README.md file. Then, run the following cells in order. The final output will allow you to interact with different parameters to see its impacts on the output.

In [1]:
INPUT = 'config2.json'

In [ ]:
import scipy as sp
import json
from raw_deface_opt import run_raw_deface, set_masks, make_A_B, compute_image, is_well_conditioned

with open(INPUT) as _f:
    _config = json.load(_f)
readout_axis = _config['input_data']['readout_axis']

nc, data, dataID, og_image_rsos = run_raw_deface('config2.json') # run raw_deface and load data

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
from ipywidgets import interact, IntSlider, Dropdown, Dropdown, Output, VBox, Button
import os
from scipy.linalg import orth
from scipy.linalg import norm
from ROVir import form_virtual_coil_data, rovir
from raw_deface_opt import set_masks, make_A_B
import numpy as np
from IPython.display import clear_output, display
import nibabel as nib

# make a save button for display image
save_button = Button(description='Save current view',button_style='success')

# make slider for selecting top virtual coils
n_coils_slider = IntSlider(value=1, min=1, 
                    max=nc, step=1, 
                    description='Number of Top Virtual Coils', 
                    continuous_update=False, 
                    style={'description_width': 'initial'},
                    layout={'width': '600px'})

# make sliders for x, y, z slices of the brain
x_slicer = IntSlider(value=data.shape[0]//2, min=0,
                     max = data.shape[0]-1, step=1,
                     description='Sagittal Slice',
                     continuous_update=False, 
                     style={'description_width': 'initial'},
                     layout={'width': '600px'})

y_slicer = IntSlider(value=data.shape[1]//2, min=0,
                     max = data.shape[1]-1, step=1,
                     description='Coronal Slice',
                     continuous_update=False, 
                     style={'description_width': 'initial'},
                     layout={'width': '600px'})

z_slicer = IntSlider(value=data.shape[2]//2, min=0,
                     max = data.shape[2]-1, step=1,
                     description='Axial Slice',
                     continuous_update=False, 
                     style={'description_width': 'initial'},
                     layout={'width': '600px'})

# make drop down box to select masking options
mask_selector = Dropdown(options=['Default', 'A', 'B', 'C', 'D', 'E', 'F'],
                         value='Default', description='Masking Scheme',
                         style={'description_width': 'initial'},
                         layout={'width': '300px'})

# make slider to choose gap between brain and face masks
gap_slider = IntSlider(value=10, min=0, max=30, 
                       step=1, description='Brain and Face Mask Gap',
                       continuous_update=False, 
                       style={'description_width': 'initial'},
                       layout={'width': '600px'})



In [ ]:
plot_output = Output()
save_status = Output()
cache = {} # use dictionary to cache already computed images
last_render = {}
max_cache = 10

def recompute_output(n_coils, mask_option, gap):

    cache_key = f'{dataID}_{n_coils}_{mask_option}_{gap}'
    if cache_key in cache:
        return cache[cache_key]

    # load default face and brain mask
    maskA = (nib.load(f'segmentations/output_mask_{dataID}/brain.nii.gz')).get_fdata()
    maskB = (nib.load(f'segmentations/output_mask_{dataID}/face.nii.gz')).get_fdata()

    if mask_option != 'Default':
        maskA, maskB = set_masks(maskA, maskB, mask_option, gap) # compute manipulated masks

    hybrid_fft = sp.fft.fftshift(sp.fft.ifft(sp.fft.fftshift(data, axes = (readout_axis)), axis=readout_axis, overwrite_x=True), axes = (readout_axis))

    brain_covars = [] # to store the brain covariance matrix for each slice, list of nc x nc matrices
    face_covars = [] # to store the face covariance matrix for each slice, list of nc x nc matrices
    eigenvecs = [] # to store the eigenvectors for each slice, list of nc x nc matrices
    num_top_eigenvecs = [] # to store the recommended number of top eigenvectors for each slice, list of integers
   
    # for each readout slice, find the recommended number of top eigenvectors and the eigenvectors
    for slice_num in range(data.shape[readout_axis]): # loop through slices in the readout direction
        
        slice = np.take(hybrid_fft, slice_num, axis=readout_axis)

        full_fft = sp.fft.fftshift(sp.fft.ifftn(sp.fft.fftshift(slice, axes = (0,1)), axes=(0,1), overwrite_x=True), axes = (0,1)) # get image, shape (y, z, ch)
            
        A, B = make_A_B(full_fft, nc, np.take(maskA, slice_num, axis=readout_axis), np.take(maskB, slice_num, axis=readout_axis)) # compute covariance matrices from slice image

        brain_covars.append(A) # append brain covariance matrix for current slice
        face_covars.append(B) # append face covariance matrix for current slice
    
        # check the condition of matrix B to ensure no problems are faced in eigh
        if not is_well_conditioned(B) and is_well_conditioned(A) : # if B is rank-deficient and A is not 
            eigenvec = rovir(nc, A, np.eye(nc, dtype=B.dtype)) # solve just Av = λv
        elif not is_well_conditioned(B) and not is_well_conditioned(A): # if B and A are rank-deficient
            eigenvec = np.eye(nc, dtype=A.dtype) # eigenvec for current slice is I so virtual coils for slice are original
        else: eigenvec = rovir(nc, A, B) # finding the eigenvectors for (brain_covar)v = λ(face_covar)v

        eigenvecs.append(eigenvec) # append eigenvec for current slice
    
        if "top_coils" not in inputs["coil_selection"]: # choose based on heuristics the number of eigenvectors to retain
            cur_top_eigenvec = top_nv(eigenvec, nc, A, B, method, threshold, slice_num, inputs["visualization"]["graph"])
            num_top_eigenvecs.append(cur_top_eigenvec) # append recommended number of top eigenvecs to retain

    np.save(f'results/{dataID}_eigenvecs.npy', eigenvecs)

    from collections import Counter

    if "top_coils" in inputs["coil_selection"]: # if user specifies the number of top virtual coils to keep, use that 
        top_eigenvec = inputs["coil_selection"]["top_coils"]
    else: # if using the automated version, decide on unified number of top eigenvecs to retain
        top_eigenvec = max(num_top_eigenvecs)

    print(GREEN + f'[UPDATE] The top {top_eigenvec} eigenvectors will be retained.' + RESET)

    virtual_coil_data_all_slices = [] # to store the virtual coil data for each slice, a list of (ky, kz, ch) matrices
    virtual_coil_data_all_slices_unaligned = [] # to store the virtual coil data pre-alignment for debugging 

    weighted_mean_brain_retain = 0
    weighted_mean_face_retain = 0

    eigenvecs_aligned = [orth(eigenvecs[0][:, :top_eigenvec]).conj().T]
    # for each readout slice, compute the virtual coil information 
    for slice_num in range(data.shape[readout_axis]):
        
        # get current slice hybrid fft
        if readout_axis == 0:
            slice = hybrid_fft[slice_num, :, :, :] # get current slice, shape (ky, kz, ch)
        elif readout_axis == 1: 
            slice = hybrid_fft[:, slice_num, :, :] # get current slice, shape (ky, kz, ch)
        elif readout_axis == 2:
            slice = hybrid_fft[:, :, slice_num, :] # get current slice, shape (ky, kz, ch)

        eigenvec = eigenvecs[slice_num] # load the eigenvectors for the current slice
        
        eigenvec_retain = orth((eigenvec)[:,:top_eigenvec]) # retain only the top eigenvectors, orthonormalize to noise-whiten
        # eigenvec_retain is a NxM matrix
            
        # =========== for unaligned data =======================================
        cur_virtual_coil_data_unaligned = form_virtual_coil_data((eigenvec)[:,:top_eigenvec], slice)
        # cur_virtual_coil_data_unaligned = form_virtual_coil_data(eigenvec_retain, slice)
        virtual_coil_data_all_slices_unaligned.append(cur_virtual_coil_data_unaligned)
        # ======================================================================

        # perform phase alignment, assuming first slice is aligned
        if slice_num != 0: 
            # nv x nc * nc x nv = nv x nv
            c =  eigenvec_retain.conj().T @ (eigenvecs_aligned[slice_num-1]).conj().T
            U, _, Vh = np.linalg.svd(c) # each is nv x nv
            P = Vh.conj().T @ U.conj().T
            eigenvecs_aligned.append(P @ eigenvec_retain.conj().T) 

        eigenvec_retain = eigenvecs_aligned[slice_num].conj().T
        orth_proj = eigenvec_retain @ eigenvec_retain.conj().T # find orthogonal projection matrix for span of retained eigenvectors
    
        # calculate signal retained from maskA region
        if norm(brain_covars[slice_num], ord = 'fro') != 0:
            cur_brain_retain = (norm((orth_proj @ brain_covars[slice_num] @ orth_proj), ord = 'fro') / norm(brain_covars[slice_num], ord = 'fro'))*100
        else: cur_brain_retain = 0

        # calculate signal retained from maskB region
        if norm(face_covars[slice_num], ord = 'fro') != 0:
            cur_face_retain = (norm((orth_proj @ face_covars[slice_num] @ orth_proj), ord = 'fro') / norm(face_covars[slice_num], ord = 'fro'))*100
        else: cur_face_retain = 0

        weight_brain_cur = np.sum(np.take(maskA, slice_num, axis=readout_axis)) / np.sum(maskA) 
        weight_face_cur = np.sum(np.take(maskB, slice_num, axis=readout_axis)) / np.sum(maskB)

        weighted_mean_brain_retain += weight_brain_cur * cur_brain_retain
        weighted_mean_face_retain += weight_face_cur * cur_face_retain

        # forming virtual coils, with eigenvectors as linear combo weights
        cur_virtual_coil_data = form_virtual_coil_data(eigenvec_retain, slice) # form virtual coils for this slice
        virtual_coil_data_all_slices.append(cur_virtual_coil_data)

    # ======== display virtual coils for unaligned stuff ============

    virtual_hybrid_unaligned = np.stack(virtual_coil_data_all_slices_unaligned, axis=readout_axis) # stack along the x axis to form (x, ky, kz, nv)
    del virtual_coil_data_all_slices_unaligned
    remaining_axes = tuple(ax for ax in (0, 1, 2) if ax != readout_axis) # the two axes still in k-space after the 1D readout ifft
    virtual_coil_data_unaligned = sp.fft.fftshift(sp.fft.ifftn(sp.fft.fftshift(virtual_hybrid_unaligned, axes = remaining_axes), axes=remaining_axes, overwrite_x=True), axes = remaining_axes)
    display_virtual_coils(virtual_coil_data_unaligned, 60, 'Virtual Coils Before Alignment',axis=0) # sagittal view
    display_virtual_coils(virtual_coil_data_unaligned, 60, 'Virtual Coils Before Alignment', axis=2) # axial view, matches readout_axis

    # ===============================================================
    print(GREEN + f'Mean Brain Signal Retention: {weighted_mean_brain_retain}')
    print(f'Mean Face Signal Retention: {weighted_mean_face_retain}' + RESET)

    virtual_hybrid = np.stack(virtual_coil_data_all_slices, axis=readout_axis) # stack along the x axis to form (x, ky, kz, nv)
    
    
    ratio = image_rsos/og_image_rsos # take the ratio between the defaced and original image

    result = {
        'maskA': maskA,
        'maskB': maskB,
        'image_rsos': image_rsos,
        'ratio': ratio,
        'brain_retain': brain_retain,
        'face_retain': face_retain,
        'eigenvec_retain': eigenvec_retain,
        'virtual_coil_data': virtual_coil_data
    }

    cache[cache_key] = result

    # clear least recent cache
    if len(cache) > max_cache:
        oldest_key = next(iter(cache))
        removed = cache.pop(oldest_key)
    
    return result

def reslice():
    n_coils = n_coils_slider.value
    mask_option = mask_selector.value
    gap = gap_slider.value 
    x, y, z = x_slicer.value, y_slicer.value, z_slicer.value

    with plot_output:
        clear_output(wait=True)
        display('RECOMPUTING DEFACED IMAGES...')

    with save_status:
        clear_output(wait=True)

    save_button.disabled = True
    
    # recompute output or get from cache
    result = recompute_output(n_coils, mask_option, gap)

    # unpack the results
    maskA = result['maskA']
    maskB = result['maskB']
    image_rsos = result['image_rsos']
    ratio = result['ratio']
    brain_retain = result['brain_retain']
    face_retain = result['face_retain']

    with plot_output:
        clear_output(wait=True)
        plt.close('all')

        # plot original, mask overlayed, and defaced images
        fig = plt.figure(figsize=(15, 10))
    
        plt.subplot(4,3,1)
        plt.imshow(np.rot90(og_image_rsos[:, :, z]), cmap='gray')
    
        plt.subplot(4,3,2)
        plt.imshow(np.rot90(og_image_rsos[x, :, :]), cmap='gray')
    
        plt.subplot(4,3,3)
        plt.imshow(np.rot90(og_image_rsos[:, y, :]), cmap='gray')
    
        plt.subplot(4,3,4)
        plt.imshow(np.rot90(og_image_rsos[:, :, z]), cmap='gray')
        plt.imshow(np.rot90(maskB[:, :, z]), alpha = 0.3, cmap = 'Reds')
        plt.imshow(np.rot90(maskA[:, :, z]), alpha = 0.3, cmap = 'Greens')
        
        plt.subplot(4,3,5)
        plt.imshow(np.rot90(og_image_rsos[x, :, :]), cmap='gray')
        plt.imshow(np.rot90(maskB[x, :, :]), alpha = 0.3, cmap = 'Reds')
        plt.imshow(np.rot90(maskA[x, :, :]), alpha = 0.3, cmap = 'Greens')
        
        plt.subplot(4,3,6)
        plt.imshow(np.rot90(og_image_rsos[:, y, :]), cmap='gray')
        plt.imshow(np.rot90(maskB[:, y, :]), alpha = 0.3, cmap = 'Reds')
        plt.imshow(np.rot90(maskA[:, y, :]), alpha = 0.3, cmap = 'Greens')
    
        plt.subplot(4,3,7)
        plt.imshow(np.rot90(image_rsos[:, :, z]), cmap='gray')
    
        plt.subplot(4,3,8)
        plt.imshow(np.rot90(image_rsos[x, :, :]), cmap='gray')
    
        plt.subplot(4,3,9)
        plt.imshow(np.rot90(image_rsos[:, y, :]), cmap='gray')
        
        global_vmin = np.min([ratio[:, :, z].min(), ratio[x, :, :].min(), ratio[:, y, :].min()])
        global_vmax = np.max([ratio[:, :, z].max(), ratio[x, :, :].max(), ratio[:, y, :].max()])
    
        plt.subplot(4,3,10)
        ratio_img1 = plt.imshow(np.rot90(ratio[:, :, z]), cmap='RdYlGn', vmin=global_vmin, vmax=global_vmax)
    
        plt.subplot(4,3,11)
        ratio_img2 = plt.imshow(np.rot90(ratio[x, :, :]), cmap='RdYlGn', vmin=global_vmin, vmax=global_vmax)
    
        plt.subplot(4,3,12)
        ratio_img3 = plt.imshow(np.rot90(ratio[:, y, :]), cmap='RdYlGn', vmin=global_vmin, vmax=global_vmax)
    
        ratio_colourbar_ax = plt.axes([0.25, 0.05, 0.5, 0.015]) 
        ratio_colourbar = plt.colorbar(ratio_img3, cax=ratio_colourbar_ax, orientation='horizontal')
        ratio_colourbar.set_label('Retention Ratio', fontsize=10)
    
        plt.show()
    
        print(f'Brain region signal retention: {brain_retain}')
        print(f'Face region signal retention: {face_retain}')

    last_render['fig'] = fig
    last_render['params'] = (n_coils, mask_option, gap, x, y, z)
    save_button.disabled = False

In [ ]:
def on_slider_change(change):
    reslice()

def on_save_clicked(b):
    if 'fig' not in last_render:
        return

    os.makedirs('results', exist_ok=True)
    n_coils, mask_option, gap, x, y, z = last_render['params']
    filename = f"results/{dataID}_ncoils{n_coils}_mask{mask_option}_gap{gap}_x{x}_y{y}_z{z}.png"
    last_render['fig'].savefig(filename, dpi=150, bbox_inches='tight')

    with save_status:
        clear_output(wait=True)
        print(f'Saved to {filename}')

save_button.on_click(on_save_clicked)
    
for widget in (n_coils_slider, mask_selector, gap_slider, x_slicer, y_slicer, z_slicer):
    widget.observe(on_slider_change, names='value')

reslice() # initial images generation 

# display widgets onto the screen
display(VBox([n_coils_slider, mask_selector, gap_slider, x_slicer, y_slicer, z_slicer, plot_output, save_button, save_status]))